# 04 — Build Your First MCP Server (FastMCP)

We've spent three notebooks on the *agent* side. Now we cross over to **MCP** — the protocol that standardises how agents talk to tools and data sources.

**Mental model in one line:** MCP is the USB-C port for AI applications. Same plug shape on both ends, regardless of who built the AI app or who built the data source.

**By the end of this notebook you will:**
1. Build a working MCP server with **FastMCP** in ~50 lines
2. Inspect it with the **MCP Inspector** (the `curl` of MCP development)
3. Connect a Python client and run the full lifecycle (initialize → list_tools → call_tool)
4. See the real JSON-RPC 2.0 traffic that flows under the hood


## 1. Why FastMCP

Before FastMCP, you'd hand-write JSON-RPC 2.0 message handlers, manage `tools/list` and `tools/call` dispatch, define schemas in JSON. With FastMCP:

- A `@mcp.tool()` decorator on a Python function exposes it as a tool
- Type hints (`title: str`) become the input JSON Schema
- Docstring becomes the tool description
- Args section in the docstring becomes per-parameter descriptions
- The lifecycle messages (`initialize`, `tools/list`, etc.) are auto-generated

FastMCP started as a standalone library and was absorbed into the official `mcp` Python SDK. We use the SDK-bundled version: `from mcp.server.fastmcp import FastMCP`.


## 2. The notes server

Lives at `src/deepbrief/mcp_servers/notes_server.py`. It exposes four tools for the research agent to persist findings during a brief.

Read it first.

In [ ]:
import inspect
from deepbrief.mcp_servers import notes_server

# Show just the top of the file plus the four tool functions
src = inspect.getsource(notes_server)
print(src[:1200])
print("... [docstrings + tool bodies — see file] ...")

## 3. What FastMCP auto-generates

For each `@mcp.tool()` function FastMCP introspects:
- Function name → tool name
- Type hints → JSON Schema for `inputSchema`
- Docstring summary → tool description
- Docstring `Args:` block → per-parameter `description`

Let's verify by listing what the FastMCP instance would expose.

In [ ]:
# FastMCP keeps its registered tools internally; we can ask the SDK for the list
tools = await notes_server.mcp.list_tools()
for t in tools:
    print(f"• {t.name}")
    print(f"  description: {(t.description or '').splitlines()[0]}")
    print(f"  schema:      {t.inputSchema}")
    print()

Notice the schemas have proper types and descriptions — all from your Python type hints + docstring. No JSON-Schema-by-hand. This is the FastMCP win.


## 4. Run it as a real subprocess and connect a client

Now we'll launch the server as a *subprocess* (stdio transport) and connect from Python. This is exactly how Claude Desktop and Cursor talk to MCP servers when the server lives on the same machine.

We use the official MCP SDK's `stdio_client` + `ClientSession`.

In [ ]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Tell the SDK how to spawn our server
server_params = StdioServerParameters(
    command=sys.executable,
    args=["-m", "deepbrief.mcp_servers.notes_server"],   # stdio default
)

# Open the transport, then the session — both are async context managers
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        # PHASE 1: handshake — capability negotiation happens here
        init_result = await session.initialize()
        print(f"Connected to: {init_result.serverInfo.name} v{init_result.serverInfo.version}")
        print(f"Server capabilities: {init_result.capabilities}\n")

        # PHASE 2: discovery
        tools = (await session.list_tools()).tools
        print(f"Tools advertised by server ({len(tools)}):")
        for t in tools:
            print(f"  • {t.name}")

        # PHASE 3: invocation
        save_result = await session.call_tool(
            "save_note",
            {"title": "WebGPU support", "content": "Stable in Chrome 121+ ([source](https://...))"},
        )
        print(f"\nsave_note → {save_result.content[0].text}")

        list_result = await session.call_tool("list_notes", {})
        print(f"list_notes → {list_result.content[0].text}")

What happened on the wire:

```jsonc
// 1. We sent (over stdin):
{ "jsonrpc": "2.0", "id": 1, "method": "initialize",
  "params": { "protocolVersion": "2025-11-25", "capabilities": {...}, "clientInfo": {...} } }

// 2. Server replied (over stdout):
{ "jsonrpc": "2.0", "id": 1,
  "result": { "protocolVersion": "2025-11-25", "capabilities": {...},
               "serverInfo": { "name": "deepbrief-notes", "version": "..." } } }

// 3. We sent the initialized notification, then list_tools, then call_tool — same pattern.
```

FastMCP wrote all that for you. You wrote four Python functions.


## 5. Debug with the MCP Inspector (the `curl` of MCP)

When something doesn't work, you don't want to `print()` your way through async transport code. You want a **GUI** that shows you the exact JSON-RPC traffic and lets you call `tools/list` and `tools/call` by hand.

That's the MCP Inspector. Run this in a terminal (NOT in the notebook):

```bash
npx @modelcontextprotocol/inspector python -m deepbrief.mcp_servers.notes_server
```

It opens **http://localhost:6274** with:
- A **History** tab — every JSON-RPC request and response
- A **Tools** tab — list + call buttons (auto-renders forms from your schemas)
- A **Resources** / **Prompts** tab — empty in our case (we only expose tools)

Try this drill:

1. Click **Connect**
2. **Tools** tab → **List Tools** → you should see `save_note`, `list_notes`, `read_note`, `delete_note`
3. Click `save_note` → fill `title` and `content` → **Run Tool**
4. Click `list_notes` → **Run Tool** → see your note
5. Open the **History** tab and verify each call sent valid JSON-RPC 2.0

Source: official inspector tool — `npm i -g @modelcontextprotocol/inspector` for permanent install.

## 6. Why we exposed *tools*, not *resources* or *prompts*

MCP servers can expose three primitives:

| Primitive | REST analogy | Who controls it |
|---|---|---|
| **Tools** | `POST` — has side effects, may compute | **Model-controlled** |
| **Resources** | `GET` — read-only | **Application-controlled** |
| **Prompts** | reusable templates | **User-controlled** |

We used tools for everything because the LLM needs to *decide* when to save and read notes. If we wanted to expose, say, *all existing notes* as read-only context the host can prefetch, we'd register them as resources. We don't need that yet.

**Practical rule:** start with tools. Add resources only when you have read-only state the host should prefetch. Add prompts only if your host has a slash-command UI (Claude Desktop does).

## 7. Self-check

1. What does FastMCP generate from a Python function's type hints? From its docstring?
2. What are the three protocol phases an MCP client goes through after connecting?
3. Why is the **MCP Inspector** the equivalent of `curl` for MCP development?
4. Name the three primitives an MCP server can expose, and who controls invocation of each.

## What's next

Notebook **05** — same server, **change one line** to expose it as Streamable HTTP. We'll explore stdio vs HTTP transports, why Plain SSE was deprecated in 2025, and the `Mcp-Session-Id` header.